# MNIST 分类的量子机器学习

## 概述

本教程的目的不是从机器学习的角度来更好地设计用于 MNIST 分类的量子机器学习方法。相反，我们使用一个简单的参数化电路，并演示 ``tensorcircuit``的量子机器学习相关的技术组件。此外，这个 jupyter notebook 绝不代表是量子机器学习的好的实践。
[WIP note]

## 设置

In [1]:
from functools import partial
import numpy as np
import tensorflow as tf
import jax
from jax import config

config.update("jax_enable_x64", True)
from jax import numpy as jnp
import optax
import tensorcircuit as tc

In [2]:
tc.set_backend("tensorflow")
tc.set_dtype("complex128")

('complex128', 'float64')

## 数据处理

我们利用 MNIST 数据并将它们调整为 3*3 以适应 9 量子位电路。
我们使用的测试平台是二进制分类任务，区分数字``1``与``5``。
由于本教程不是关于量子机器学习的良好实践，因此我们将验证集放在一边。
我们只为一个小型演示收集 100 个数据点用于实验。

In [3]:
# numpy 数据
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train[..., np.newaxis] / 255.0


def filter_pair(x, y, a, b):
    keep = (y == a) | (y == b)
    x, y = x[keep], y[keep]
    y = y == a
    return x, y


x_train, y_train = filter_pair(x_train, y_train, 1, 5)
x_train_small = tf.image.resize(x_train, (3, 3)).numpy()
x_train_bin = np.array(x_train_small > 0.5, dtype=np.float32)
x_train_bin = np.squeeze(x_train_bin)[:100]

In [4]:
# tensorflow 数据

x_train_tf = tf.reshape(tf.constant(x_train_bin, dtype=tf.float64), [-1, 9])
y_train_tf = tf.constant(y_train[:100], dtype=tf.float64)

# jax 数据

x_train_jax = jnp.array(x_train_bin, dtype=np.float64).reshape([100, -1])
y_train_jax = jnp.array(y_train[:100], dtype=np.float64).reshape([100])

## 使用 ``vectorized_value_and_grad`` API

In [5]:
nlayers = 3


def qml_loss(x, y, weights, nlayers):
    n = 9
    weights = tc.backend.cast(weights, "complex128")
    x = tc.backend.cast(x, "complex128")
    c = tc.Circuit(n)
    for i in range(n):
        c.rx(i, theta=x[i])
    for j in range(nlayers):
        for i in range(n - 1):
            c.cnot(i, i + 1)
        for i in range(n):
            c.rx(i, theta=weights[2 * j, i])
            c.ry(i, theta=weights[2 * j + 1, i])
    ypred = c.expectation([tc.gates.z(), (4,)])
    ypred = tc.backend.real(ypred)
    ypred = (tc.backend.real(ypred) + 1) / 2.0
    return -y * tc.backend.log(ypred) - (1 - y) * tc.backend.log(1 - ypred), ypred

In [6]:
def get_qml_vvag():
    qml_vvag = tc.backend.vectorized_value_and_grad(
        qml_loss, argnums=(2,), vectorized_argnums=(0, 1), has_aux=True
    )
    qml_vvag = tc.backend.jit(qml_vvag, static_argnums=(3,))
    return qml_vvag


qml_vvag = get_qml_vvag()
qml_vvag(x_train_tf, y_train_tf, tf.ones([nlayers * 2, 9], dtype=tf.float64), nlayers)

((<tf.Tensor: shape=(100,), dtype=float64, numpy=
  array([0.8433698 , 0.56257199, 0.54653163, 0.56257199, 0.82036163,
         0.56257199, 0.56257199, 0.58030506, 0.82036163, 0.56257199,
         0.82036163, 0.56257199, 0.82036163, 0.56257199, 0.54653163,
         0.54653163, 0.56257199, 0.56257199, 0.58030506, 0.82036163,
         0.54653163, 0.56257199, 0.56257199, 0.56257199, 0.56257199,
         0.56257199, 0.56257199, 0.85182866, 0.56257199, 0.82036163,
         0.82036163, 0.56257199, 0.8433698 , 0.56257199, 0.8433698 ,
         0.56257199, 0.85182866, 0.56257199, 0.82036163, 0.54653163,
         0.56257199, 0.56257199, 0.56257199, 0.56257199, 0.8433698 ,
         0.58030506, 0.56257199, 0.82036163, 0.8433698 , 0.8433698 ,
         0.54653163, 0.56257199, 0.82036163, 0.86501404, 0.56257199,
         0.56257199, 0.8433698 , 0.56257199, 0.85182866, 0.82036163,
         0.82036163, 0.56257199, 0.82036163, 0.56257199, 0.56257199,
         0.56257199, 0.82036163, 0.8433698 , 0.843369

In [7]:
# %timeit qml_vvag(x_train_tf, y_train_tf, tf.ones([nlayers*2, 9], dtype=tf.float64), nlayers)

### Jax 后端兼容性

In [8]:
tc.set_backend("jax")

jax_backend

In [9]:
qml_vvag = get_qml_vvag()
qml_vvag(
    x_train_jax, y_train_jax, jnp.ones([nlayers * 2, 9], dtype=np.float64), nlayers
)

((Array([0.8433698 , 0.56257199, 0.54653163, 0.56257199, 0.82036163,
         0.56257199, 0.56257199, 0.58030506, 0.82036163, 0.56257199,
         0.82036163, 0.56257199, 0.82036163, 0.56257199, 0.54653163,
         0.54653163, 0.56257199, 0.56257199, 0.58030506, 0.82036163,
         0.54653163, 0.56257199, 0.56257199, 0.56257199, 0.56257199,
         0.56257199, 0.56257199, 0.85182866, 0.56257199, 0.82036163,
         0.82036163, 0.56257199, 0.8433698 , 0.56257199, 0.8433698 ,
         0.56257199, 0.85182866, 0.56257199, 0.82036163, 0.54653163,
         0.56257199, 0.56257199, 0.56257199, 0.56257199, 0.8433698 ,
         0.58030506, 0.56257199, 0.82036163, 0.8433698 , 0.8433698 ,
         0.54653163, 0.56257199, 0.82036163, 0.86501404, 0.56257199,
         0.56257199, 0.8433698 , 0.56257199, 0.85182866, 0.82036163,
         0.82036163, 0.56257199, 0.82036163, 0.56257199, 0.56257199,
         0.56257199, 0.82036163, 0.8433698 , 0.8433698 , 0.82036163,
         0.56257199, 0.56257199, 0

In [10]:
# %timeit qml_vvag(x_train_jax, y_train_jax, jnp.ones([nlayers * 2, 9], dtype=np.float64), nlayers)

### 使用 ``tf.data`` 训练模型

In [11]:
# 转换回 tensorflow
tc.set_backend("tensorflow")
qml_vvag = get_qml_vvag()
qml_vvag = tc.backend.jit(qml_vvag, static_argnums=(3,))

In [12]:
mnist_data = (
    tf.data.Dataset.from_tensor_slices((x_train_tf, y_train_tf))
    .repeat(200)
    .shuffle(100)
    .batch(32)
)

In [13]:
opt = tf.keras.optimizers.Adam(1e-2)
w = tf.Variable(
    initial_value=tf.random.normal(shape=(2 * nlayers, 9), stddev=0.5, dtype=tf.float64)
)
for i, (xs, ys) in zip(range(2000), mnist_data):
    (losses, ypreds), grad = qml_vvag(xs, ys, w, nlayers)
    if i % 20 == 0:
        print(tf.reduce_mean(losses))
    opt.apply_gradients([(grad[0], w)])

tf.Tensor(0.6856853925292129, shape=(), dtype=float64)
tf.Tensor(0.6235271307895434, shape=(), dtype=float64)


tf.Tensor(0.6364645943683445, shape=(), dtype=float64)
tf.Tensor(0.6112911720348524, shape=(), dtype=float64)


tf.Tensor(0.6831693219087261, shape=(), dtype=float64)
tf.Tensor(0.6271953575462331, shape=(), dtype=float64)


tf.Tensor(0.5604584018083277, shape=(), dtype=float64)
tf.Tensor(0.5737426326383851, shape=(), dtype=float64)


tf.Tensor(0.5537529468894505, shape=(), dtype=float64)
tf.Tensor(0.6064862339410262, shape=(), dtype=float64)


...

tf.Tensor(0.6027745860165212, shape=(), dtype=float64)
tf.Tensor(0.5828273190054151, shape=(), dtype=float64)


tf.Tensor(0.5889524404979345, shape=(), dtype=float64)
tf.Tensor(0.614848290780694, shape=(), dtype=float64)


tf.Tensor(0.6501886762223215, shape=(), dtype=float64)
tf.Tensor(0.5855507482005314, shape=(), dtype=float64)


tf.Tensor(0.6974819563048922, shape=(), dtype=float64)
tf.Tensor(0.6328383946375984, shape=(), dtype=float64)


tf.Tensor(0.5863845189501637, shape=(), dtype=float64)
tf.Tensor(0.7294857389456698, shape=(), dtype=float64)


## 使用 ``tf.keras`` API

In [14]:
from tensorcircuit import keras


def qml_y(x, weights, nlayers):
    n = 9
    weights = tc.backend.cast(weights, "complex128")
    x = tc.backend.cast(x, "complex128")
    c = tc.Circuit(n)
    for i in range(n):
        c.rx(i, theta=x[i])
    for j in range(nlayers):
        for i in range(n - 1):
            c.cnot(i, i + 1)
        for i in range(n):
            c.rx(i, theta=weights[2 * j, i])
            c.ry(i, theta=weights[2 * j + 1, i])
    ypred = c.expectation([tc.gates.z(), (4,)])
    ypred = tc.backend.real(ypred)
    ypred = (tc.backend.real(ypred) + 1) / 2.0
    return ypred


ql = keras.QuantumLayer(partial(qml_y, nlayers=nlayers), [(2 * nlayers, 9)])

In [15]:
# 带有 value and grad 范例的 keras 接口


@tf.function
def my_vvag(xs, ys):
    with tf.GradientTape() as tape:
        ypred = ql(xs)
        loss = tf.keras.losses.BinaryCrossentropy()(ys, ypred)
    return loss, tape.gradient(loss, ql.variables)


my_vvag(x_train_tf, y_train_tf)

(<tf.Tensor: shape=(), dtype=float32, numpy=0.6920610666275024>,
 [<tf.Tensor: shape=(6, 9), dtype=float64, numpy=
  array([[-1.13219932e-02, -1.60624207e-03, -1.20717572e-02,
           2.73360172e-03, -3.48446692e-03,  1.27177631e-03,
           9.50652270e-05,  9.10729825e-18,  2.16840434e-18],
         [ 3.20507710e-03, -3.51588859e-03, -1.60489670e-02,
           3.98961424e-03,  1.06845007e-02,  5.14626744e-03,
          -9.18889766e-03,  3.46944695e-18, -5.31259065e-18],
         [-1.76178807e-03, -2.90488238e-03,  8.09255145e-03,
          -8.87927796e-03, -9.00375845e-03,  3.45542704e-04,
          -3.03576608e-18,  7.48099499e-18,  4.77048956e-18],
         [ 7.50661140e-03,  1.56883026e-02,  6.59044375e-03,
          -8.09964381e-03,  6.28331773e-03,  3.85381662e-03,
           7.37257477e-18,  1.12757026e-17, -4.77048956e-18],
         [ 3.79470760e-18,  9.54097912e-18,  1.30104261e-17,
          -1.30104261e-18,  1.94108414e-03,  8.67361738e-19,
           9.10729825e-18, 

In [16]:
# %timeit my_vvag(x_train_tf, y_train_tf)

In [17]:
# keras 接口与 keras 训练范例

model = tf.keras.Sequential([ql])

model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(0.01),
    metrics=[tf.keras.metrics.BinaryAccuracy()],
)

model.fit(x_train_tf, y_train_tf, batch_size=32, epochs=100)

Epoch 1/100


1/4 ━━━━━━━━━━━━━━━━━━━━ 55s 18s/step - binary_accuracy: 0.5938 - loss: 0.7125

4/4 ━━━━━━━━━━━━━━━━━━━━ 18s 15ms/step - binary_accuracy: 0.6300 - loss: 0.6913


Epoch 2/100


1/4 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - binary_accuracy: 0.5938 - loss: 0.7043

...

1/4 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - binary_accuracy: 0.7188 - loss: 0.5954

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - binary_accuracy: 0.6300 - loss: 0.6165


Epoch 100/100


1/4 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - binary_accuracy: 0.5312 - loss: 0.6689

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - binary_accuracy: 0.6300 - loss: 0.6150


### Keras 中的量子经典混合模型

In [18]:
def qml_ys(x, weights, nlayers):
    n = 9
    weights = tc.backend.cast(weights, "complex128")
    x = tc.backend.cast(x, "complex128")
    c = tc.Circuit(n)
    for i in range(n):
        c.rx(i, theta=x[i])
    for j in range(nlayers):
        for i in range(n - 1):
            c.cnot(i, i + 1)
        for i in range(n):
            c.rx(i, theta=weights[2 * j, i])
            c.ry(i, theta=weights[2 * j + 1, i])
    ypreds = []
    for i in range(n):
        ypred = c.expectation([tc.gates.z(), (i,)])
        ypred = tc.backend.real(ypred)
        ypred = (tc.backend.real(ypred) + 1) / 2.0
        ypreds.append(ypred)
    return tc.backend.stack(ypreds)

In [19]:
ql = tc.keras.QuantumLayer(partial(qml_ys, nlayers=nlayers), [(2 * nlayers, 9)])
model = tf.keras.Sequential([ql, tf.keras.layers.Dense(1, activation="sigmoid")])

In [20]:
model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(0.01),
    metrics=[tf.keras.metrics.BinaryAccuracy()],
)

model.fit(x_train_tf, y_train_tf, batch_size=32, epochs=100)

Epoch 1/100


1/4 ━━━━━━━━━━━━━━━━━━━━ 1:11 24s/step - binary_accuracy: 0.5625 - loss: 0.7011

4/4 ━━━━━━━━━━━━━━━━━━━━ 24s 22ms/step - binary_accuracy: 0.6300 - loss: 0.6700


Epoch 2/100


1/4 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - binary_accuracy: 0.7188 - loss: 0.6238

...

1/4 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - binary_accuracy: 0.7812 - loss: 0.5365

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - binary_accuracy: 0.7400 - loss: 0.5637


Epoch 100/100


1/4 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - binary_accuracy: 0.6875 - loss: 0.5704

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - binary_accuracy: 0.7400 - loss: 0.5628


### Jax 中的混合模型

In [21]:
tc.set_backend("jax")

jax_backend

In [22]:
key = jax.random.PRNGKey(42)
key, *subkeys = jax.random.split(key, num=4)
params = {
    "qweights": jax.random.normal(subkeys[0], shape=[nlayers * 2, 9]),
    "cweights:w": jax.random.normal(subkeys[1], shape=[9]),
    "cweights:b": jax.random.normal(subkeys[2], shape=[1]),
}

In [23]:
def qml_hybrid_loss(x, y, params, nlayers):
    weights = params["qweights"]
    w = params["cweights:w"]
    b = params["cweights:b"]
    ypred = qml_ys(x, weights, nlayers)
    ypred = tc.backend.reshape(ypred, [-1, 1])
    ypred = w @ ypred + b
    ypred = jax.nn.sigmoid(ypred)
    ypred = ypred[0]
    loss = -y * tc.backend.log(ypred) - (1 - y) * tc.backend.log(1 - ypred)
    return loss

In [24]:
qml_hybrid_loss_vag = tc.backend.jit(
    tc.backend.vvag(qml_hybrid_loss, vectorized_argnums=(0, 1), argnums=2),
    static_argnums=3,
)

In [25]:
qml_hybrid_loss_vag(x_train_jax, y_train_jax, params, nlayers)

(Array([0.42296214, 1.06451071, 1.06809695, 1.06451071, 0.42617817,
        0.99293054, 1.06451071, 1.05843045, 0.42617817, 0.99293054,
        0.42617817, 1.06451071, 0.42617817, 1.06451071, 1.06809695,
        1.00047892, 1.06451071, 1.06451071, 1.05843045, 0.42617817,
        1.00047892, 1.06451071, 1.06451071, 0.99293054, 1.06451071,
        1.06451071, 0.99293054, 0.41962206, 1.06451071, 0.47539413,
        0.42617817, 0.99293054, 0.42296214, 0.99293054, 0.42296214,
        1.06451071, 0.46151499, 1.06451071, 0.42617817, 1.06809695,
        1.06451071, 1.06451071, 0.99293054, 0.99293054, 0.42296214,
        0.97190916, 0.99293054, 0.47539413, 0.42296214, 0.46281253,
        1.06809695, 0.99293054, 0.42617817, 0.45839653, 1.06451071,
        1.06451071, 0.42296214, 1.06451071, 0.41962206, 0.47539413,
        0.42617817, 0.99293054, 0.42617817, 1.06451071, 0.99293054,
        1.06451071, 0.47539413, 0.46281253, 0.46281253, 0.42617817,
        1.06451071, 1.06451071, 1.06451071, 1.06

In [26]:
optimizer = optax.adam(5e-3)
opt_state = optimizer.init(params)
for i, (xs, ys) in zip(range(2000), mnist_data):  # 在这里使用 tensorflow 数据加载器
    xs = xs.numpy()
    ys = ys.numpy()
    v, grads = qml_hybrid_loss_vag(xs, ys, params, nlayers)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    if i % 30 == 0:
        print(jnp.mean(v))

0.8181925262433516


0.697530656565215


0.5781537353438815


0.6608575982192403


0.6017298954531876


...

0.5878545684901912


0.5210577599290486


0.593101390005848


0.4801157315597049


0.5133667752589907
